In [2]:
import glob
import pandas as pd
import numpy as np
from pathlib import Path

# ── Configuration ───────────────────────────────────────────────────
PROJECT_ROOT = Path('/home/h604827/ControlActions')
INPUT_DIR  = PROJECT_ROOT / 'DATA/25_tags_events'
OUTPUT_DIR = PROJECT_ROOT / 'DATA/25_tags_events_preprocessed'
TRIP_CSV   = PROJECT_ROOT / 'DATA/Final_List_Trip_Duration.csv'
TIME_OFFSET = pd.Timedelta(hours=1.5)
DEDUP_COLS  = ['VT_Start', 'Source', 'ConditionName', 'Description']
CHUNK_SIZE  = 100_000

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Load trip periods (once) ────────────────────────────────────────
trip_data = pd.read_csv(TRIP_CSV)
trip_data['Stop Date']  = pd.to_datetime(trip_data['Stop Date'])
trip_data['Start Date'] = pd.to_datetime(trip_data['Start Date'])
stop_dates  = trip_data['Stop Date'].values
start_dates = trip_data['Start Date'].values

# ── Discover UUID folders ──────────────────────────────────────────
uuid_dirs = sorted([d for d in INPUT_DIR.iterdir() if d.is_dir()])
print(f"Found {len(uuid_dirs)} UUID folders in {INPUT_DIR}")

# ── Process each UUID folder ───────────────────────────────────────
summary = []

for uuid_dir in uuid_dirs:
    uuid_name = uuid_dir.name

    # Find all _E.parquet part files under this UUID folder
    parquet_files = glob.glob(
        str(uuid_dir / '2021011814_2025062803_E.parquet' / 'S=*' / '*.parquet')
    )
    if not parquet_files:
        print(f"  [{uuid_name}] No _E parquet files found – skipping")
        continue

    # 1. Read and concatenate all part files
    dfs = [pd.read_parquet(f) for f in parquet_files]
    df = pd.concat(dfs, ignore_index=True)
    n_raw = len(df)

    # 2. Adjust VT_Start by -1.5 hours
    if 'VT_Start' in df.columns:
        df['VT_Start'] = pd.to_datetime(df['VT_Start'], format='mixed') - TIME_OFFSET
    else:
        print(f"  [{uuid_name}] WARNING: VT_Start column not found – skipping time offset")

    df = df.sort_values('VT_Start').reset_index(drop=True)

    # 3. Remove events during trip periods (chunked broadcasting)
    event_times = df['VT_Start'].values
    in_trips = np.zeros(len(event_times), dtype=bool)

    for i in range(0, len(event_times), CHUNK_SIZE):
        chunk = event_times[i:i + CHUNK_SIZE]
        in_range = (chunk[:, None] >= stop_dates) & (chunk[:, None] <= start_dates)
        in_trips[i:i + CHUNK_SIZE] = in_range.any(axis=1)

    n_trip = int(in_trips.sum())
    df = df[~in_trips].reset_index(drop=True)

    # 4. Deduplicate rows
    n_pre_dedup = len(df)
    df = df.groupby(DEDUP_COLS, sort=False).first().reset_index()
    df = df.sort_values('VT_Start').reset_index(drop=True)
    n_deduped = n_pre_dedup - len(df)

    # 5. Save preprocessed parquet
    out_path = OUTPUT_DIR / f'{uuid_name}.parquet'
    df.to_parquet(out_path, index=False)

    summary.append({
        'uuid': uuid_name,
        'raw_rows': n_raw,
        'trip_removed': n_trip,
        'duplicates_removed': n_deduped,
        'final_rows': len(df),
        'sources': df['Source'].nunique() if 'Source' in df.columns else None
    })
    print(f"  [{uuid_name}] {n_raw} → {len(df)} rows "
          f"(trips: -{n_trip}, dupes: -{n_deduped}) → {out_path.name}")

# ── Summary ────────────────────────────────────────────────────────
summary_df = pd.DataFrame(summary)
print(f"\n{'='*70}")
print(f"Preprocessed {len(summary_df)} UUID folders → {OUTPUT_DIR}/")
print(f"Total: {summary_df['raw_rows'].sum():,} raw → {summary_df['final_rows'].sum():,} final rows")
print(summary_df.to_string(index=False))

Found 11 UUID folders in /home/h604827/ControlActions/DATA/25_tags_events
  [0102a4fc-aae1-414b-a790-e62529880ede] 110212 → 54540 rows (trips: -11197, dupes: -44475) → 0102a4fc-aae1-414b-a790-e62529880ede.parquet
  [032761f5-dbdf-4d83-9566-498b49fa05ee] 220374 → 124600 rows (trips: -11290, dupes: -84484) → 032761f5-dbdf-4d83-9566-498b49fa05ee.parquet
  [3ad21a42-9b2f-450a-91f3-f78b9cb4e99e] 91578 → 31166 rows (trips: -18806, dupes: -41606) → 3ad21a42-9b2f-450a-91f3-f78b9cb4e99e.parquet
  [572ec917-c98e-4932-9cc6-eacd831be962] 124569 → 68510 rows (trips: -5070, dupes: -50989) → 572ec917-c98e-4932-9cc6-eacd831be962.parquet
  [6b9de746-f4ab-4082-acfa-75bf21530bbf] 1051748 → 906230 rows (trips: -96538, dupes: -48980) → 6b9de746-f4ab-4082-acfa-75bf21530bbf.parquet
  [8c7c9aeb-3c57-4e87-8ee0-5e33c0499e8d] 523768 → 313410 rows (trips: -142979, dupes: -67379) → 8c7c9aeb-3c57-4e87-8ee0-5e33c0499e8d.parquet
  [a74a1df6-3231-4c91-be6a-3179156c1572] 24298 → 14467 rows (trips: -5419, dupes: -4412) 